# Tune3 — Projeto Completo (Colab A100)

Projeto montado e testado (68 testes). Faça upload do `tune3_PROJETO_COMPLETO.zip` ou clone do seu GitHub.

**Ordem:** setup -> dataset -> testes -> experimentos (piloto / S1 / S2).

## 1. Setup do projeto

In [ ]:
# OPCAO A: upload do zip
from google.colab import files
up = files.upload()   # selecione tune3_PROJETO_COMPLETO.zip
import zipfile, os
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z: z.extractall('/content/')
os.chdir('/content/tune3_FULL' if os.path.exists('/content/tune3_FULL') else '/content')
print('cwd:', os.getcwd())

# OPCAO B (alternativa): clonar do seu GitHub
# !git clone https://github.com/SEU_USUARIO/tune3.git /content/tune3 && %cd /content/tune3

In [ ]:
!pip install -e . -q
!pip install botorch gpytorch statsmodels -q
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU')

## 2. Dataset DREBIN-215

In [ ]:
import os; os.makedirs('data', exist_ok=True)
if not os.path.exists('data/drebin215.csv'):
    from google.colab import files
    up = files.upload()   # selecione drebin215.csv
    import shutil; shutil.move(list(up.keys())[0], 'data/drebin215.csv')
from tune3.data.drebin import DrebinLoader, DrebinConfig
Xtr,Xv,Xte,ytr,yv,yte = DrebinLoader(DrebinConfig(csv_path='data/drebin215.csv')).load_splits()
print(f'treino={Xtr.shape} val={Xv.shape} teste={Xte.shape} frac_malware={ytr.mean():.3f}')
assert Xtr.shape[1]==215; print('Dataset OK')

## 3. Testes (sanidade)

In [ ]:
!pytest tune3/tests/ -q -m 'not slow'   # rapido; tire o -m para os 68 testes

## 4a. Piloto in-distribution (H1) — confirma o resultado base

In [ ]:
!python scripts/run_pilot_parallel.py --csv data/drebin215.csv \
  --seeds 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 \
  --epochs 100 --n-init 8 --n-iter 20 --device cuda --workers 4 \
  --out confirma_bestcvar_20seeds.json

## 4b. S1 — Desbalanceamento (H2) ~6h

In [ ]:
!python scripts/run_s1_imbalance.py --csv data/drebin215.csv \
  --seeds 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 \
  --ratios 0.05 0.02 0.01 --epochs 100 --device cuda --out s1_imbalance.json

## 4c. S2 — Zero-day por cluster (H3) ~5h

In [ ]:
!python scripts/run_s2_zeroday.py --csv data/drebin215.csv \
  --seeds 0 1 2 3 4 5 6 7 8 9 --n-clusters 5 \
  --epochs 100 --device cuda --out s2_zeroday.json

## 5. Baixar resultados

In [ ]:
from google.colab import files
for f in ['confirma_bestcvar_20seeds.json','s1_imbalance.json','s2_zeroday.json']:
    import os
    if os.path.exists(f): files.download(f)